## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. It also installs `scikit-learn`, which this analysis needs, into the same session. You can read every line of the code below before you run it.

# Snack market segments: k-means with four clusters

**Cluster Analysis** · a worked example from the StatNex teaching guide

**The story.** A consumer panel scored 56 snacks on 13 attributes. From these, each snack has three summary scores, centred so that 0 is an average snack: how indulgent it is, how healthy it seems, and how much value it gives. A marketing team wants four market segments.

**The question.** What four segments does the snack market split into?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `09_multivariate/cluster_analysis__kmeans.csv` (56 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Snack,Indulgence score,Health score,Value score
Sea Salt Waves,0.53,-1.085,1.393
BBQ Crunchers,-0.592,-0.394,1.048
Cheesy Puffs,0.357,-1.085,0.703
Sour Cream Ridges,-0.333,0.066,-0.505
Truffle Kettle Chips,-0.419,-0.509,-0.678
Chili Lime Tortillas,0.185,-0.164,0.357
Classic Pretzels,-0.592,-0.164,1.307
Wasabi Peas,0.444,-0.625,-0.333
Smoked Almonds,0.875,-0.279,0.012
Butter Popcorn,-0.937,-0.74,1.307
Parmesan Crisps,-0.16,-0.625,-0.678
Seaweed Snacks,-0.247,-1.43,0.444
Pizza Bites,-0.247,-0.855,1.134
Rosemary Crackers,-0.333,0.181,-0.678
Spicy Ramen Chips,-0.247,-0.74,1.393
Salted Cashews,-0.764,0.526,-0.851
Cheddar Crackers,-0.592,-0.97,1.134
Onion Rings Snack,-1.109,-0.164,0.789
Choco Crunch Bar,1.048,0.066,0.53
Caramel Cluster,0.271,0.526,-0.074
Dark Chocolate 70%,0.961,-0.394,-0.851
Milk Choco Buttons,0.789,-0.279,1.22
Gummy Bears,0.961,-0.049,0.875
Sour Worms,0.444,-0.394,-0.16
Hazelnut Wafers,1.048,-0.74,0.789
Salted Caramel Truffles,1.048,-0.164,-0.678
Double Choc Cookies,1.134,-1.085,-0.16
Mini Donuts,0.703,-0.394,0.271
Marshmallow Twists,0.099,-0.855,0.875
Matcha Kit Bar,1.22,0.526,-1.541
Fudge Brownie Bites,1.134,-0.279,0.271
Strawberry Licorice,0.616,0.181,0.703
Belgian Pralines,0.961,0.526,-1.109
Peanut Butter Cups,0.444,0.526,-0.505
Cookie Dough Bites,0.789,-0.049,0.961
Honey Waffles,1.565,-0.97,-0.16
Jelly Beans,0.444,0.641,0.875
Choc Chip Muffin,0.789,0.756,-0.16
Macaron Box,0.789,-0.279,-1.109
Oat Granola Bar,-0.764,0.526,-0.074
Protein Bar,-0.678,0.526,-1.282
Trail Mix,-0.592,-0.049,-0.592
Rice Cakes,-0.505,0.641,0.185
Dried Mango,-0.678,0.296,-0.333
Veggie Straws,-1.455,0.641,0.875
Apple Chips,-0.851,0.641,0.185
Greek Yogurt Bites,-0.851,0.526,-1.282
Chickpea Crisps,-1.109,1.331,-1.109
Kale Chips,-1.109,0.526,-0.419
Fruit Leather,-1.196,0.871,0.185
Almond Butter Bites,-0.851,1.101,-1.196
Banana Chips,-0.505,0.526,1.048
Seed Crackers,-0.592,1.331,-1.455
Coconut Clusters,-0.333,0.641,-1.282
Berry Oat Bites,-0.592,0.756,-0.333
Energy Balls,-0.419,0.411,-1.282
""")

## Helper: kmeans_start

A step that no library does, written out. In the code download it is the file `kmeans_start.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""The deterministic starting centres for k-means.

k-means only finds a local optimum, so where it starts decides the answer. Starting
from the first k cases would be arbitrary; this rule scans the cases once and keeps k
cases that are spread far apart:

* the first k cases are provisional centres;
* a later case replaces one of the two closest centres (whichever is nearer to the
  case) when the case is further from every centre than those two are from each
  other;
* failing that, it replaces its own nearest centre when its second-nearest centre is
  further away than that nearest centre's own nearest neighbour.
"""

import numpy as np


def initial_centres(x, k):
    centres = x[:k].copy()
    for case in x[k:]:
        between = np.sqrt(((centres[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(between, np.inf)
        a, b = np.unravel_index(np.argmin(between), between.shape)  # the closest pair
        neighbour = between.min(axis=1)
        to_case = np.sqrt(((centres - case) ** 2).sum(axis=1))
        nearest, second = np.argsort(to_case, kind="stable")[:2]  # a tie: the first
        if to_case[nearest] > between[a, b]:
            centres[a if to_case[a] < to_case[b] else b] = case
        elif to_case[second] > neighbour[nearest]:
            centres[nearest] = case
    return centres

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from sklearn.metrics import silhouette_samples
# initial_centres: defined in the helper cell kmeans_start above (kmeans_start.py in the code download)

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells count as missing, and a case missing any variable is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Indulgence score", "Health score", "Value score"]
label_column = ["Snack"]
numbers = data[variables].apply(pd.to_numeric, errors="coerce")
complete = numbers.notna().all(axis=1)
x = numbers[complete].to_numpy(float)
n, p = x.shape
k = 4
# Each case is named by its label column, or numbered 1, 2, ... among the complete
# cases when there is none.
names = (data.loc[complete, label_column[0]].astype(str).tolist() if label_column
         else [str(i + 1) for i in range(n)])

summary = pd.DataFrame({"n": [n, len(data) - n, len(data)]},
                       index=["Valid cases", "Incomplete cases", "Total"])
summary["share"] = summary["n"] / len(data)
print(summary)

## Descriptive statistics

In [ ]:
# Variance and standard deviation divide by n - 1. A variable with a much larger SD
# than the others weighs more in a Euclidean distance.
describe = pd.DataFrame(
    {"n": n, "mean": x.mean(axis=0), "variance": x.var(axis=0, ddof=1),
     "SD": x.std(axis=0, ddof=1), "min": x.min(axis=0), "max": x.max(axis=0)},
    index=variables,
)
print()
print(describe)
correlation = np.corrcoef(x, rowvar=False)
print()
print(pd.DataFrame(correlation, index=variables, columns=variables).round(4))

## k-means: starting centres

In [ ]:
# k-means only finds a local optimum, so the start decides the answer; the rule in
# kmeans_start.py picks k real cases that lie far apart (the rule SPSS's QUICK CLUSTER
# uses), so the same data always gives the same clusters.
# Euclidean distance on the variables as they are: a scale of 1 for each.
scale = np.ones(p)
scaled = x / scale
clusters = [str(c + 1) for c in range(k)]
start = initial_centres(scaled, k)
initial = pd.DataFrame((start * scale).T, index=variables, columns=clusters)
print()
print(initial)

## k-means

In [ ]:
# scikit-learn's KMeans with Lloyd's algorithm, the one StatNex (and Stata) use: assign
# every case to its nearest centre, move each centre to the mean of its cases, and
# repeat until no case changes cluster. init= hands it the starting centres above and
# n_init=1 stops it trying random starts of its own; tol=0 means it stops only when
# nothing moves. Lloyd's algorithm stops at the first split that this cannot improve,
# which need not be the tightest split there is: Hartigan and Wong's algorithm (R's
# default) also moves single cases and can end elsewhere from the same start.
# inertia_ is the within-cluster sum of squares the algorithm tries to make small.
model = KMeans(n_clusters=k, init=start, n_init=1, max_iter=20,
               tol=0.0, algorithm="lloyd").fit(scaled)
centres = model.cluster_centers_
assignment = model.labels_
print()
print(f"Passes: {model.n_iter_}; within-cluster sum of squares: {model.inertia_:.4f}")

## How far each centre moved

In [ ]:
# KMeans reports where the centres ended, not the path they took, so the two steps
# are repeated here from the same start to list how far each centre moved on each
# pass. All zeros means converged; the last row reproduces KMeans's centres.
path = start.copy()
shifts = []
for iteration in range(1, 20 + 1):
    nearest = cdist(scaled, path).argmin(axis=1)
    updated = np.array([scaled[nearest == c].mean(axis=0) if (nearest == c).any() else path[c]
                        for c in range(k)])
    moved = np.sqrt(((updated - path) ** 2).sum(axis=1))
    shifts.append([iteration, *moved])
    path = updated
    if moved.max() == 0:
        break
shifts = pd.DataFrame(shifts, columns=["iteration", *clusters])
print()
print(shifts.to_string(index=False))

## Final centres

In [ ]:
# Each cluster's mean on every variable: read them to name the clusters. Then the
# distances between the centres: two close centres may not be worth separating.
final = pd.DataFrame((centres * scale).T, index=variables, columns=clusters)
between = cdist(centres, centres)
print()
print(final)
print(pd.DataFrame(between, index=clusters, columns=clusters))

## One-way ANOVA of each variable across the clusters

In [ ]:
# statsmodels' anova_lm on a one-way model per variable. A large F says the clusters
# differ a lot on that variable, so it did much of the separating. The P values are
# not tests: the clusters were chosen to make these F values large, so P is small
# by construction even for data with no real groups. Both sums of squares are taken
# around each cluster's own mean, so between + within = total even when the
# iteration cap stopped k-means before its centres settled.
rows = []
for j, name in enumerate(variables):
    one = pd.DataFrame({"value": x[:, j], "cluster": assignment + 1})
    table = anova_lm(ols("value ~ C(cluster)", data=one).fit())
    effect, error = table.iloc[0], table.iloc[1]
    rows.append([effect["sum_sq"], int(effect["df"]), effect["mean_sq"],
                 error["sum_sq"], int(error["df"]), error["mean_sq"], effect["F"], effect["PR(>F)"]])
anova = pd.DataFrame(rows, index=variables,
                     columns=["SS between", "df", "MS", "SS within", "df", "MS", "F", "P"])
print()
print(anova.to_string())

## Cluster sizes and membership

In [ ]:
# KMeans's transform() gives each case's distance to every final centre; its cluster
# is the nearest one.
size = np.bincount(assignment, minlength=k)
counts = pd.DataFrame([[*size, n], [*(size / n), 1.0]], index=["n", "%"], columns=[*clusters, "total"])
distance = model.transform(scaled)
membership = pd.DataFrame(distance, columns=clusters, index=names)
membership.insert(0, "cluster", assignment + 1)
membership.insert(0, "No.", np.arange(1, n + 1))
print()
print(counts)
print(membership.round(4).to_string())

## Silhouette widths

In [ ]:
# How well the final partition holds together, measured on the distances it was built
# from (k-means: Euclidean between the cases, on the scale k-means used). An object's
# width sets its mean distance a to the rest of its own cluster against its mean
# distance b to the nearest other cluster, s = (b - a) / max(a, b): near 1 it sits
# firmly in its cluster, below 0 the neighbouring cluster is closer. An object alone
# in its cluster counts as 0 (Rousseeuw 1987); scikit-learn's silhouette_samples
# computes them, as R's cluster::silhouette does. Kaufman and Rousseeuw (1990) read
# the overall mean: above 0.70 strong, above 0.50 reasonable, above 0.25 weak (could
# be artificial), else no substantial structure. These are rules of thumb, not tests.
width = silhouette_samples(scaled, assignment, metric="euclidean")
group = assignment + 1
silhouette = pd.DataFrame(
    {"size": [*((group == c).sum() for c in range(1, k + 1)), len(group)],
     "mean width": [*(width[group == c].mean() if (group == c).any() else np.nan
                      for c in range(1, k + 1)), width.mean()]},
    index=[*(f"Cluster {c}" for c in range(1, k + 1)), "Total"])
overall = width.mean()
if overall > 0.70:
    rating = "A strong structure"
elif overall > 0.50:
    rating = "A reasonable structure"
elif overall > 0.25:
    rating = "A weak structure (could be artificial)"
else:
    rating = "No substantial structure"
print()
print(silhouette.round(4))
print("Overall:", rating)

## What the result says

K-means clustering splits cases into a number of groups that you choose, here four. It moves each cluster centre until no snack changes group; the centres stopped moving at iteration 5.

The final cluster centres describe the segments. Cluster 1 (19 snacks) is indulgent and good value: everyday treats. Cluster 2 (9 snacks) has the highest value score, about 1.1: value savoury snacks. Cluster 3 (8 snacks) is the most indulgent, 0.81, but low on value: premium indulgence. Cluster 4 (20 snacks) has the highest health score, 0.49: better-for-you premium.

But the mean silhouette width, a score of how well cases sit in their own cluster, is only 0.33: a weak structure. So the four segments are useful labels, but the snacks form one continuous spread. The segments are regions of it, not separate islands.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.